# TURN / RIVER RAISE pass — headless GPU

Adds a **Raise** option to the full-range turn/river pack: re-solves the 16 curated runouts with `--raise-x 3`, so facing-a-bet spots on the turn/river gain Fold/**Call**/**Raise** (currently Fold/Call only). First-to-act & checked-to spots stay Check/Bet.

The raise tree is bigger than the plain full-range pass, so budget **~1–3 h** (still one commit, well under Kaggle's ~9 h limit). If it ever runs long, drop `--n` to `300`.

**One commit:** Accelerator → GPU T4 x2, Internet On, Save & Run All. Download `flop_pack_turnriver_fullrange.db` (+ `.gz`) and send them back — they drop straight into the trainer (raise-enabled superset), replacing the current turn/river pack.

*(Still unconditioned — pre-flop SRP ranges with card-removal only, NOT filtered by a prior check/check line.)*

In [ ]:
# Fail fast if no GPU.
import subprocess
try:
    _r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit('No nvidia-smi (CPU instance) -- set Accelerator -> GPU, then re-run.')
assert _r.returncode == 0 and 'GPU' in _r.stdout, 'NO GPU -- set Accelerator -> GPU T4 x2, then re-run.'
print(_r.stdout.strip())

In [ ]:
# Clone the solver source (needs Internet On).
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
print('source ready')

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

In [ ]:
# Solve all 16 runouts at full range WITH raises on GPU (~1-3 h).
import subprocess, os
env = {**os.environ, 'PYTHONPATH': 'src'}
subprocess.run(
    ['python', 'demo/gen_turn_river.py',
     '--solver', 'gpu', '--dtype', 'float32', '--n', '400', '--iters', '300',
     '--raise-x', '3',
     '--version', 'turnriver_fullrange',
     '--note', 'turn/river full range + raise (NOT check-check filtered)'],
    cwd='/kaggle/working/poker', env=env, check=True)


In [ ]:
# Expose the pack for download.
import shutil, os
src='/kaggle/working/poker/output/packs/flop_pack_turnriver_fullrange.db'
if not os.path.exists(src):
    raise SystemExit('No turnriver_fullrange pack -- check the solve cell above.')
shutil.copy(src, '/kaggle/working/flop_pack_turnriver_fullrange.db')
shutil.copy(src+'.gz', '/kaggle/working/flop_pack_turnriver_fullrange.db.gz')
print('DOWNLOAD -> flop_pack_turnriver_fullrange.db (%d KB) + .gz'
      % (os.path.getsize('/kaggle/working/flop_pack_turnriver_fullrange.db')//1024))